In [2]:
pip install pandas numpy plotly pycountry

   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   --- ------------------------------------ 0.8/9.9 MB 5.5 MB/s eta 0:00:02
   -------- ------------------------------- 2.1/9.9 MB 5.9 MB/s eta 0:00:02
   ------------- -------------------------- 3.4/9.9 MB 6.2 MB/s eta 0:00:02
   --------------------- ------------------ 5.2/9.9 MB 6.8 MB/s eta 0:00:01
   ------------------------------- -------- 7.9/9.9 MB 8.0 MB/s eta 0:00:01
   ---------------------------------------- 9.9/9.9 MB 8.5 MB/s  0:00:01
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ------------------- -------------------- 3.9/8.0 MB 19.1 MB/s eta 0:00:01
   ------------------------------------- -- 7.6/8.0 MB 18.7 MB/s eta 0:00:01
   ---------------------------------------- 8.0/8.0 MB 16.5 MB/s  0:00:00

   ---------------------------------------- 0/3 [pycountry]
   ---------------------------------------- 0/3 [pycountry]
   ---------------------------------------- 0/3 [pycountry]


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd

# Load the Piece 1 output file
FILE = "open_router_data_classified.csv"   # change if needed
df = pd.read_csv(FILE)

print("=== Basic shape ===")
print(df.shape)

print("\n=== Columns ===")
print(df.columns.tolist())

# --------------------------------------------------
# 1. Overall classification counts
# --------------------------------------------------
print("\n=== classification_source counts ===")
print(df["classification_source"].value_counts(dropna=False))

# --------------------------------------------------
# 2. Flag counts
# --------------------------------------------------
flag_cols = [
    "is_open_source",
    "is_chinese_model",
    "is_unknown_country",
    "is_unclassified_model",
    "is_missing_model_slug",
]

print("\n=== Flag counts ===")
for col in flag_cols:
    if col in df.columns:
        print(f"{col}: {df[col].fillna(False).sum()}")

# --------------------------------------------------
# 3. Rows and token share by classification status
# --------------------------------------------------
summary = pd.DataFrame({
    "rows": df["classification_source"].value_counts(dropna=False),
    "tokens": df.groupby("classification_source", dropna=False)["total_tokens"].sum()
}).fillna(0)

summary["row_pct"] = (summary["rows"] / len(df) * 100).round(2)
summary["token_pct"] = (summary["tokens"] / df["total_tokens"].sum() * 100).round(2)

print("\n=== Rows and token share by classification_source ===")
print(summary.sort_values("tokens", ascending=False))

# --------------------------------------------------
# 4. Unique model counts
# --------------------------------------------------
model_summary = (
    df.groupby("classification_source", dropna=False)["model_permaslug"]
      .nunique()
      .rename("unique_models")
      .reset_index()
      .sort_values("unique_models", ascending=False)
)

print("\n=== Unique model slugs by classification_source ===")
print(model_summary)

# --------------------------------------------------
# 5. Top unclassified models by rows and tokens
# --------------------------------------------------
unclassified = df[df["classification_source"] == "unclassified"].copy()

if len(unclassified) == 0:
    print("\n=== No unclassified models found ===")
else:
    top_unclassified = (
        unclassified.groupby("model_permaslug", dropna=False)
        .agg(
            rows=("model_permaslug", "size"),
            total_tokens=("total_tokens", "sum"),
            countries=("country", "nunique"),
            months=("month", "nunique"),
        )
        .reset_index()
        .sort_values(["total_tokens", "rows"], ascending=[False, False])
    )

    print("\n=== Top unclassified models ===")
    print(top_unclassified.head(50))

# --------------------------------------------------
# 6. Top missing / unknown data issues
# --------------------------------------------------
print("\n=== Unknown country rows ===")
unknown_country = df[df["is_unknown_country"] == True]
print(len(unknown_country))

print("\n=== Missing model slug rows ===")
missing_slug = df[df["is_missing_model_slug"] == True]
print(len(missing_slug))

# --------------------------------------------------
# 7. Quick monthly view of classified vs unclassified
# --------------------------------------------------
monthly_status = (
    df.assign(
        status=df["classification_source"].where(
            df["classification_source"].isin(["unclassified", "missing_slug"]),
            "classified"
        )
    )
    .groupby(["month", "status"], dropna=False)
    .agg(
        rows=("status", "size"),
        total_tokens=("total_tokens", "sum")
    )
    .reset_index()
    .sort_values(["month", "status"])
)

print("\n=== Monthly classified vs unclassified summary ===")
print(monthly_status.head(100))

# --------------------------------------------------
# 8. Optional: save review tables
# --------------------------------------------------
summary.to_csv("classification_summary.csv")
model_summary.to_csv("classification_unique_models.csv", index=False)
monthly_status.to_csv("classification_monthly_status.csv", index=False)

if len(unclassified) > 0:
    top_unclassified.to_csv("top_unclassified_models.csv", index=False)

print("\nSaved review files:")
print("- classification_summary.csv")
print("- classification_unique_models.csv")
print("- classification_monthly_status.csv")
if len(unclassified) > 0:
    print("- top_unclassified_models.csv")

=== Basic shape ===
(385932, 10)

=== Columns ===
['month', 'model_permaslug', 'country', 'total_tokens', 'is_open_source', 'is_chinese_model', 'classification_source', 'is_unknown_country', 'is_unclassified_model', 'is_missing_model_slug']

=== classification_source counts ===
classification_source
unclassified       256228
family_fallback     82536
exact_lookup        47168
Name: count, dtype: int64

=== Flag counts ===
is_open_source: 101176
is_chinese_model: 75025
is_unknown_country: 4019
is_unclassified_model: 256228
is_missing_model_slug: 0

=== Rows and token share by classification_source ===
                         rows           tokens  row_pct  token_pct
classification_source                                             
unclassified           256228  176754718643133    66.39      74.59
family_fallback         82536   41988039862888    21.39      17.72
exact_lookup            47168   18233904888112    12.22       7.69

=== Unique model slugs by classification_source ===
  cl

In [ ]:
import pandas as pd

# Load the Piece 1 output file
FILE = "open_router_data_classified.csv"   # change if needed
df = pd.read_csv(FILE)

print("=== Basic shape ===")
print(df.shape)

print("\n=== Columns ===")
print(df.columns.tolist())

# --------------------------------------------------
# 1. Overall classification counts
# --------------------------------------------------
print("\n=== classification_source counts ===")
print(df["classification_source"].value_counts(dropna=False))

# --------------------------------------------------
# 2. Flag counts
# --------------------------------------------------
flag_cols = [
    "is_open_source",
    "is_chinese_model",
    "is_unknown_country",
    "is_unclassified_model",
    "is_missing_model_slug",
]

print("\n=== Flag counts ===")
for col in flag_cols:
    if col in df.columns:
        print(f"{col}: {df[col].fillna(False).sum()}")

# --------------------------------------------------
# 3. Rows and token share by classification status
# --------------------------------------------------
summary = pd.DataFrame({
    "rows": df["classification_source"].value_counts(dropna=False),
    "tokens": df.groupby("classification_source", dropna=False)["total_tokens"].sum()
}).fillna(0)

summary["row_pct"] = (summary["rows"] / len(df) * 100).round(2)
summary["token_pct"] = (summary["tokens"] / df["total_tokens"].sum() * 100).round(2)

print("\n=== Rows and token share by classification_source ===")
print(summary.sort_values("tokens", ascending=False))

# --------------------------------------------------
# 4. Unique model counts
# --------------------------------------------------
model_summary = (
    df.groupby("classification_source", dropna=False)["model_permaslug"]
      .nunique()
      .rename("unique_models")
      .reset_index()
      .sort_values("unique_models", ascending=False)
)

print("\n=== Unique model slugs by classification_source ===")
print(model_summary)

# --------------------------------------------------
# 5. Top unclassified models by rows and tokens
# --------------------------------------------------
unclassified = df[df["classification_source"] == "unclassified"].copy()

if len(unclassified) == 0:
    print("\n=== No unclassified models found ===")
else:
    top_unclassified = (
        unclassified.groupby("model_permaslug", dropna=False)
        .agg(
            rows=("model_permaslug", "size"),
            total_tokens=("total_tokens", "sum"),
            countries=("country", "nunique"),
            months=("month", "nunique"),
        )
        .reset_index()
        .sort_values(["total_tokens", "rows"], ascending=[False, False])
    )

    print("\n=== Top unclassified models ===")
    print(top_unclassified.head(50))

# --------------------------------------------------
# 6. Top missing / unknown data issues
# --------------------------------------------------
print("\n=== Unknown country rows ===")
unknown_country = df[df["is_unknown_country"] == True]
print(len(unknown_country))

print("\n=== Missing model slug rows ===")
missing_slug = df[df["is_missing_model_slug"] == True]
print(len(missing_slug))

# --------------------------------------------------
# 7. Quick monthly view of classified vs unclassified
# --------------------------------------------------
monthly_status = (
    df.assign(
        status=df["classification_source"].where(
            df["classification_source"].isin(["unclassified", "missing_slug"]),
            "classified"
        )
    )
    .groupby(["month", "status"], dropna=False)
    .agg(
        rows=("status", "size"),
        total_tokens=("total_tokens", "sum")
    )
    .reset_index()
    .sort_values(["month", "status"])
)

print("\n=== Monthly classified vs unclassified summary ===")
print(monthly_status.head(100))

# --------------------------------------------------
# 8. Optional: save review tables
# --------------------------------------------------
summary.to_csv("classification_summary.csv")
model_summary.to_csv("classification_unique_models.csv", index=False)
monthly_status.to_csv("classification_monthly_status.csv", index=False)

if len(unclassified) > 0:
    top_unclassified.to_csv("top_unclassified_models.csv", index=False)

print("\nSaved review files:")
print("- classification_summary.csv")
print("- classification_unique_models.csv")
print("- classification_monthly_status.csv")
if len(unclassified) > 0:
    print("- top_unclassified_models.csv")

=== Basic shape ===
(385932, 10)

=== Columns ===
['month', 'model_permaslug', 'country', 'total_tokens', 'is_open_source', 'is_chinese_model', 'classification_source', 'is_unknown_country', 'is_unclassified_model', 'is_missing_model_slug']

=== classification_source counts ===
classification_source
unclassified       256228
family_fallback     82536
exact_lookup        47168
Name: count, dtype: int64

=== Flag counts ===
is_open_source: 101176
is_chinese_model: 75025
is_unknown_country: 4019
is_unclassified_model: 256228
is_missing_model_slug: 0

=== Rows and token share by classification_source ===
                         rows           tokens  row_pct  token_pct
classification_source                                             
unclassified           256228  176754718643133    66.39      74.59
family_fallback         82536   41988039862888    21.39      17.72
exact_lookup            47168   18233904888112    12.22       7.69

=== Unique model slugs by classification_source ===
  cl

In [11]:
pip install nbformat

  Using cached attrs-25.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached referencing-0.37.0-py3-none-any.whl.metadata (2.8 kB)
Using cached attrs-25.4.0-py3-none-any.whl (67 kB)
Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl (18 kB)
Using cached referencing-0.37.0-py3-none-any.whl (26 kB)

   ---------------------------------------- 0/7 [fastjsonschema]
   ---------------------------------------- 0/7 [fastjsonschema]
   ----------- ---------------------------- 2/7 [attrs]
   ----------- ---------------------------- 2/7 [attrs]
   ----------- ---------------------------- 2/7 [attrs]
   ----------------- ---------------------- 3/7 [referencing]
   ----------------- ---------------------- 3/7 [referencing]
   ---------------------- ----------------- 4/7 [jsonschema-specifications]
   ---------------------------- ----------- 5/7 [jsonschema]
   ---------------------------- -----------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.express as px
import pycountry

# ----------------------------
# Parameters
# ----------------------------
INPUT_FILE = "country_month_indices.csv"
OUTPUT_DIR = Path("piece4_outputs")
TOP_N_COUNTRIES_FOR_LINES = 15
EXCLUDE_UNKNOWN_COUNTRY = True

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------
# Load and validate
# ----------------------------
df = pd.read_csv(INPUT_FILE)

required_cols = {
    "country",
    "month",
    "total_tokens",
    "token_index",
    "open_source_index",
    "chinese_index",
}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

# ----------------------------
# Clean fields
# ----------------------------
df = df.copy()

df["country"] = (
    df["country"]
    .fillna("UNK")
    .astype(str)
    .str.strip()
    .str.upper()
    .replace("", "UNK")
)

df["month"] = df["month"].fillna("").astype(str).str.strip()

unique_months = pd.Series(df["month"].unique(), name="month")
month_lookup = pd.Series(
    pd.to_datetime(unique_months, errors="coerce").dt.to_period("M").values,
    index=unique_months
)
df["month_period"] = df["month"].map(month_lookup)

for col in ["total_tokens", "token_index", "open_source_index", "chinese_index"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["total_tokens"] = df["total_tokens"].fillna(0)
df.loc[df["total_tokens"] < 0, "total_tokens"] = 0

for col in ["token_index", "open_source_index", "chinese_index"]:
    df[col] = df[col].fillna(0).clip(lower=0, upper=100)

df = df.dropna(subset=["month_period"]).copy()

if EXCLUDE_UNKNOWN_COUNTRY:
    df = df.loc[df["country"] != "UNK"].copy()

if df.empty:
    raise ValueError("No rows available for visualisation after cleaning/filtering.")

df = df.sort_values(["month_period", "country"]).reset_index(drop=True)
df["month_label"] = df["month_period"].astype(str)

# ----------------------------
# ISO3 helper
# ----------------------------
def iso2_to_iso3(code: str):
    code = (code or "").strip().upper()

    manual = {
        "UK": "GBR",
        "EL": "GRC",
        "XK": "XKX",
    }
    if code in manual:
        return manual[code]

    if len(code) == 3 and code.isalpha():
        return code

    if len(code) != 2 or not code.isalpha():
        return None

    match = pycountry.countries.get(alpha_2=code)
    if match:
        return match.alpha_3

    fallback = {
        "US": "USA", "GB": "GBR", "DE": "DEU", "FR": "FRA", "IT": "ITA",
        "ES": "ESP", "PT": "PRT", "NL": "NLD", "BE": "BEL", "CH": "CHE",
        "AT": "AUT", "SE": "SWE", "NO": "NOR", "DK": "DNK", "FI": "FIN",
        "IE": "IRL", "PL": "POL", "CZ": "CZE", "HU": "HUN", "RO": "ROU",
        "BG": "BGR", "GR": "GRC", "TR": "TUR", "UA": "UKR", "RU": "RUS",
        "CA": "CAN", "MX": "MEX", "BR": "BRA", "AR": "ARG", "CL": "CHL",
        "CO": "COL", "PE": "PER", "AU": "AUS", "NZ": "NZL", "JP": "JPN",
        "CN": "CHN", "HK": "HKG", "TW": "TWN", "IN": "IND", "ID": "IDN",
        "MY": "MYS", "SG": "SGP", "TH": "THA", "VN": "VNM", "PH": "PHL",
        "KR": "KOR", "ZA": "ZAF", "NG": "NGA", "KE": "KEN", "GH": "GHA",
        "ET": "ETH", "TZ": "TZA", "UG": "UGA", "MW": "MWI", "ZM": "ZMB",
        "ZW": "ZWE", "MZ": "MOZ", "BW": "BWA", "NA": "NAM", "EG": "EGY",
        "MA": "MAR", "TN": "TUN", "DZ": "DZA", "AE": "ARE", "SA": "SAU",
        "IL": "ISR",
    }
    return fallback.get(code)

df["iso3"] = df["country"].map(iso2_to_iso3)

map_df = df.dropna(subset=["iso3"]).copy()
map_df = map_df.sort_values(["month_label", "country"]).reset_index(drop=True)

top_countries = (
    df.groupby("country", dropna=False)["total_tokens"]
    .sum()
    .sort_values(ascending=False)
    .head(TOP_N_COUNTRIES_FOR_LINES)
    .index.tolist()
)

line_df = df.loc[df["country"].isin(top_countries)].copy()
line_df = line_df.sort_values(["month_period", "country"]).reset_index(drop=True)

if line_df.empty:
    raise ValueError("No countries available for line charts after selecting top countries.")

country_iso_lookup = (
    df[["country", "iso3"]]
    .drop_duplicates()
    .sort_values(["country", "iso3"], na_position="last")
)

print("Cleaned shape:", df.shape)
print("Line chart countries:", top_countries)
print("Countries with ISO3:", map_df["country"].nunique() if not map_df.empty else 0)

Cleaned shape: (2847, 18)
Line chart countries: ['US', 'DE', 'SG', 'BE', 'NL', 'KR', 'GB', 'JP', 'IN', 'FR', 'CA', 'FI', 'HK', 'BR', 'AU']
Countries with ISO3: 238


In [2]:
def make_line_chart(data: pd.DataFrame, y_col: str, title: str, y_axis_title: str):
    fig = px.line(
        data,
        x="month_label",
        y=y_col,
        color="country",
        markers=True,
        title=title,
        hover_data={
            "country": True,
            "month_label": True,
            "total_tokens": ":,.0f",
            "token_index": ":.2f",
            "open_source_index": ":.2f",
            "chinese_index": ":.2f",
        },
        category_orders={"month_label": sorted(data["month_label"].unique())},
    )

    fig.update_layout(
        xaxis_title="Month",
        yaxis_title=y_axis_title,
        legend_title_text="Country",
        hovermode="x unified",
    )
    return fig

fig_token = make_line_chart(
    line_df,
    y_col="token_index",
    title=f"Token volume index over time — top {TOP_N_COUNTRIES_FOR_LINES} countries by total tokens",
    y_axis_title="Token volume index (0–100)",
)

fig_open = make_line_chart(
    line_df,
    y_col="open_source_index",
    title=f"Open-source index over time — top {TOP_N_COUNTRIES_FOR_LINES} countries by total tokens",
    y_axis_title="Open-source index (0–100)",
)

fig_chinese = make_line_chart(
    line_df,
    y_col="chinese_index",
    title=f"Chinese model index over time — top {TOP_N_COUNTRIES_FOR_LINES} countries by total tokens",
    y_axis_title="Chinese index (0–100)",
)

fig_token.show()
fig_open.show()
fig_chinese.show()

fig_token.write_html(OUTPUT_DIR / "line_token_index.html", include_plotlyjs="cdn")
fig_open.write_html(OUTPUT_DIR / "line_open_source_index.html", include_plotlyjs="cdn")
fig_chinese.write_html(OUTPUT_DIR / "line_chinese_index.html", include_plotlyjs="cdn")

print("Saved line charts.")

Saved line charts.


In [3]:
def make_choropleth(data: pd.DataFrame, value_col: str, title: str, colorbar_title: str):
    fig = px.choropleth(
        data,
        locations="iso3",
        color=value_col,
        hover_name="country",
        animation_frame="month_label",
        color_continuous_scale="Viridis",
        range_color=(0, 100),
        title=title,
        hover_data={
            "country": True,
            "month_label": True,
            "total_tokens": ":,.0f",
            "token_index": ":.2f",
            "open_source_index": ":.2f",
            "chinese_index": ":.2f",
            "iso3": False,
        },
        category_orders={"month_label": sorted(data["month_label"].unique())},
    )

    fig.update_layout(
        geo=dict(showframe=False, showcoastlines=True),
        coloraxis_colorbar_title=colorbar_title,
    )
    return fig

if map_df.empty:
    print("No ISO3 country mappings available, so maps were not created.")
else:
    fig_map_token = make_choropleth(
        map_df,
        value_col="token_index",
        title="Token volume index by country and month",
        colorbar_title="Token index",
    )

    fig_map_open = make_choropleth(
        map_df,
        value_col="open_source_index",
        title="Open-source index by country and month",
        colorbar_title="Open-source index",
    )

    fig_map_chinese = make_choropleth(
        map_df,
        value_col="chinese_index",
        title="Chinese model index by country and month",
        colorbar_title="Chinese index",
    )

    fig_map_token.show()
    fig_map_open.show()
    fig_map_chinese.show()

    fig_map_token.write_html(OUTPUT_DIR / "map_token_index.html", include_plotlyjs="cdn")
    fig_map_open.write_html(OUTPUT_DIR / "map_open_source_index.html", include_plotlyjs="cdn")
    fig_map_chinese.write_html(OUTPUT_DIR / "map_chinese_index.html", include_plotlyjs="cdn")

    print("Saved map charts.")

Saved map charts.


In [4]:
line_df.to_csv(OUTPUT_DIR / "line_chart_data.csv", index=False)
map_df.to_csv(OUTPUT_DIR / "map_chart_data.csv", index=False)
country_iso_lookup.to_csv(OUTPUT_DIR / "country_iso_lookup.csv", index=False)

print("\n=== Countries missing ISO3 codes ===")
missing_iso = (
    df.loc[df["iso3"].isna(), "country"]
    .drop_duplicates()
    .sort_values()
    .tolist()
)
print(missing_iso[:50])

print("\n=== Output files ===")
for p in sorted(OUTPUT_DIR.glob("*")):
    print(p.resolve())


=== Countries missing ISO3 codes ===
['T1']

=== Output files ===
C:\Users\wb637538\open-router-analysis\notebooks\piece4_outputs\country_iso_lookup.csv
C:\Users\wb637538\open-router-analysis\notebooks\piece4_outputs\line_chart_data.csv
C:\Users\wb637538\open-router-analysis\notebooks\piece4_outputs\line_chinese_index.html
C:\Users\wb637538\open-router-analysis\notebooks\piece4_outputs\line_open_source_index.html
C:\Users\wb637538\open-router-analysis\notebooks\piece4_outputs\line_token_index.html
C:\Users\wb637538\open-router-analysis\notebooks\piece4_outputs\map_chart_data.csv
C:\Users\wb637538\open-router-analysis\notebooks\piece4_outputs\map_chinese_index.html
C:\Users\wb637538\open-router-analysis\notebooks\piece4_outputs\map_open_source_index.html
C:\Users\wb637538\open-router-analysis\notebooks\piece4_outputs\map_token_index.html
